# Periodicity Gate Injection Benchmark

This notebook benchmarks the pre-periodicity gate and downstream dip recovery on injected light curves built from the bundled ASAS-SN `.dat3` controls. It compares three configurations:

- `standard_only`: standard MALCA event scoring with the configured default baseline.
- `phase_folded_only`: run the gate, then score every trial with the phase-template baseline using the gate-selected period.
- `bifurcated_gate`: route periodic gate labels to phase-template scoring and non-periodic labels to standard scoring.

Full mode defaults to a balanced 60k-trial benchmark with a coarse `500`-period CE gate scan and `10` worker processes. The expensive `5000`-period scan is reserved for the period-search subset sweep unless you explicitly set `MALCA_PGIB_GATE_N_PERIODS=5000`. For execution checks, set `MALCA_PGIB_SMOKE=1` before running the notebook.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/malca-matplotlib")

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

In [ ]:
import pandas as pd
from IPython.display import Image, display

from malca.evaluation.periodicity_gate_injection_benchmark import (
    BenchmarkConfig,
    BENCHMARK_CLASS_ORDER,
    run_benchmark,
)

def env_bool(name: str, default: bool = False) -> bool:
    value = os.environ.get(name)
    if value is None:
        return default
    return str(value).strip().lower() in {"1", "true", "yes", "y", "on"}

def env_int(name: str, default: int) -> int:
    value = os.environ.get(name)
    return default if value is None or str(value).strip() == "" else int(value)

def env_float(name: str, default: float) -> float:
    value = os.environ.get(name)
    return default if value is None or str(value).strip() == "" else float(value)

SMOKE_MODE = env_bool("MALCA_PGIB_SMOKE", False)
RUN_TAG = os.environ.get("MALCA_PGIB_RUN_TAG")

config = BenchmarkConfig(
    bundle_lc_dir=REPO_ROOT / "output/runs/runs_march18_bundle_all/bundle_assets/lightcurves",
    output_base_dir=REPO_ROOT / "output/diagnostics/periodicity_gate_injection_benchmark",
    run_tag=RUN_TAG,
    smoke_mode=SMOKE_MODE,
    total_trials=env_int("MALCA_PGIB_TOTAL_TRIALS", 60000),
    smoke_total_trials=env_int("MALCA_PGIB_SMOKE_TRIALS", 512),
    control_sample_size=env_int("MALCA_PGIB_CONTROL_SAMPLE_SIZE", 2048),
    smoke_control_sample_size=env_int("MALCA_PGIB_SMOKE_CONTROL_SAMPLE_SIZE", 96),
    gate_n_periods=env_int("MALCA_PGIB_GATE_N_PERIODS", 500),
    smoke_gate_n_periods=env_int("MALCA_PGIB_SMOKE_GATE_N_PERIODS", 800),
    period_search_subset_size=env_int("MALCA_PGIB_PERIOD_SEARCH_SUBSET", 1000),
    smoke_period_search_subset_size=env_int("MALCA_PGIB_SMOKE_PERIOD_SEARCH_SUBSET", 64),
    workers=env_int("MALCA_PGIB_WORKERS", 10),
    trial_task_size=env_int("MALCA_PGIB_TRIAL_TASK_SIZE", 1),
    baseline_func=os.environ.get("MALCA_PGIB_BASELINE_FUNC", "gp_masked"),
    min_mag_offset=env_float("MALCA_PGIB_MIN_MAG_OFFSET", 0.1),
    cache_generated_lightcurves=env_bool("MALCA_PGIB_CACHE_LIGHTCURVES", True),
)

print("Benchmark configuration")
print(f"  smoke_mode: {config.smoke_mode}")
print(f"  trials: {config.effective_total_trials:,}")
print(f"  controls: {config.effective_control_sample_size:,}")
print(f"  baseline_func: {config.baseline_func}")
print(f"  workers: {config.workers}")
print(f"  gate_n_periods: {config.effective_gate_n_periods:,}")
print(f"  bundle_lc_dir: {config.bundle_lc_dir}")
print(f"  output_base_dir: {config.output_base_dir}")

## Run Benchmark

This cell writes all required artifacts under `output/diagnostics/periodicity_gate_injection_benchmark/<run_tag>/`:

- `trial_design.parquet`
- `trial_results.parquet`
- `gate_threshold_sweep.parquet`
- `period_search_subset_sweep.parquet`
- `summary_metrics.parquet`

In [ ]:
run = run_benchmark(config)

print(f"Run directory: {run.run_dir}")
for name in [
    "trial_design.parquet",
    "trial_results.parquet",
    "gate_threshold_sweep.parquet",
    "period_search_subset_sweep.parquet",
    "summary_metrics.parquet",
]:
    path = run.run_dir / name
    print(f"  {name}: {path.exists()}  {path}")

## Sanity Checks

In [ ]:
class_counts = run.trial_design["class_name"].value_counts().reindex(BENCHMARK_CLASS_ORDER).fillna(0).astype(int)
display(class_counts.rename("n_trials").to_frame())

required_result_columns = ["standard_detected", "phase_folded_detected", "bifurcated_detected"]
missing = [col for col in required_result_columns if col not in run.trial_results.columns]
if missing:
    raise AssertionError(f"Missing required result columns: {missing}")
if (class_counts <= 0).any():
    raise AssertionError("At least one injected class has zero trials")

default_gate_counts = pd.crosstab(run.trial_results["target_gate_label"], run.trial_results["pre_periodicity_label"])
display(default_gate_counts)

distinct_branch_loads = run.gate_threshold_sweep.loc[run.gate_threshold_sweep["scope"] == "all", "periodic_branch_load"].nunique()
print(f"Distinct threshold-sweep branch loads: {distinct_branch_loads}")
if distinct_branch_loads < 2:
    print("Warning: threshold sweep did not change branch load on this sample; increase smoke trials or gate resolution.")

## Summary Tables

In [ ]:
summary_cols = [
    "scope",
    "class_name",
    "pipeline",
    "n",
    "target_recovery",
    "false_positive_rate",
    "detection_rate",
    "gate_periodic_fraction",
    "period_usable_rate",
    "gate_periodic_recall",
    "gate_false_periodic_route_rate",
]
display(run.summary_metrics[summary_cols].sort_values(["scope", "pipeline", "class_name"]))

sweep_all = run.gate_threshold_sweep[run.gate_threshold_sweep["scope"] == "all"].copy()
display(
    sweep_all.sort_values(["target_recovery", "gate_false_periodic_route_rate"], ascending=[False, True])
    .head(15)
)

display(run.period_search_subset_sweep.sort_values(["period_usable_rate", "gate_periodic_recall"], ascending=[False, False]))

## Diagnostic Plots

In [ ]:
plot_paths = [
    run.run_dir / "plots/recovery_by_class_pipeline.png",
    run.run_dir / "plots/gate_routing_confusion.png",
    run.run_dir / "plots/recovery_vs_injection_parameters.png",
    run.run_dir / "plots/threshold_heatmap_target_recovery.png",
    run.run_dir / "plots/threshold_heatmap_gate_false_periodic_route_rate.png",
    run.run_dir / "plots/threshold_heatmap_periodic_branch_load.png",
    run.run_dir / "plots/gate_sweep_pareto.png",
    run.run_dir / "plots/representative_gate_examples.png",
]

for path in plot_paths:
    if path.exists():
        print(path)
        display(Image(filename=str(path)))
    else:
        print(f"Missing plot: {path}")